# 02 - QLoRA Fine-Tuning Pipeline (Google Colab)

### Parameter-Efficient 4-Bit Fine-Tuning for Customer Support
**Project:** Customer Support RAG Chatbot  
**Execution Environment:** Google Colab (Free T4 GPU, 16GB VRAM)  
**Primary Target:** `meta-llama/Meta-Llama-3-8B-Instruct`  
**Zero-Gate Immediate Fallback:** `Qwen/Qwen2.5-7B-Instruct` or `microsoft/Phi-3-mini-4k-instruct`

---
### Key Highlights
- Loads model in 4-bit NormalFloat (NF4) with BitsAndBytes.
- Configures LoRA with `r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]`.
- Uses `SFTTrainer` with Cosine Annealing, FP16, and Gradient Accumulation.
- Subsets dataset to 5,000 - 10,000 top pairs to complete training within < 45 minutes on Colab Free T4.
- Checkpoints every 250 steps directly to Google Drive (`/content/drive/MyDrive/checkpoints/`).
- Exports LoRA adapters, merged 16-bit weights, and quantizes GGUF (`Q4_K_M`) for local Ollama/CLI usage.



In [ ]:
# Step 1: GPU Verification & Google Drive Mount
import torch
from google.colab import drive
import os

drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not torch.cuda.is_available():
    raise SystemError("GPU not detected! Please switch Colab Runtime to GPU (T4).")

print(f"[OK] GPU Detected: {torch.cuda.get_device_name(0)}")
print(f"[OK] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"[OK] Checkpoints directory: {CHECKPOINT_DIR}")



In [ ]:
# Step 2: Install Modern Fine-Tuning Stack
!pip install -q torch transformers datasets peft bitsandbytes accelerate trl



### Step 3: Model Selection with Zero-Gate Fallback
To prevent training delays caused by Meta gated access approval, we configure an automatic zero-gate fallback model (`Qwen/Qwen2.5-7B-Instruct` or `microsoft/Phi-3-mini-4k-instruct`).



In [ ]:
import os

# Select model: Set your Hugging Face token if using gated Meta-Llama-3
HF_TOKEN = os.getenv("HF_TOKEN", "")

# Toggle between Llama 3 and zero-gate fallback
USE_LLAMA3 = False  # Set to True if you have accepted Meta-Llama-3 license on HF

if USE_LLAMA3 and HF_TOKEN:
    MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
else:
    # Zero-gate high performance fallback
    MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"[*] Active Base Model: {MODEL_ID}")



In [ ]:
# Step 4: Load 4-bit Quantized Model & Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"[*] Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN or None,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"[*] Loading model in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN or None,
    trust_remote_code=True
)
model.config.use_cache = False
print("[OK] Model successfully loaded in 4-bit VRAM.")



### Step 5: Configure LoRA (PEFT)
We inject trainable adapter ranks into the self-attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`).



In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()



### Step 6: Dataset Formatting & Tokenization
Load the cleaned dataset from Google Drive and format into instruction conversational templates.



In [ ]:
import json
from datasets import Dataset

data_path = "/content/drive/MyDrive/chatbot_data/processed/cleaned_customer_support_sample.jsonl"
samples = []

with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        samples.append(json.loads(line.strip()))

# Select top 7,500 pairs for rapid, high-quality Colab T4 execution (< 40 mins)
subset = samples[:7500]
print(f"[OK] Loaded {len(subset):,} training samples from Drive.")

SYSTEM_PROMPT = (
    "You are a professional customer support assistant representing verified enterprise brands. "
    "Provide clear, polite, and actionable solutions to customer inquiries."
)

def format_prompt(item):
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n[{item['brand']}] {item['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{item['response']}<|im_end|>"
    )
    return {"text": text}

formatted_data = [format_prompt(s) for s in subset]
dataset = Dataset.from_list(formatted_data)
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(f"[OK] Training set: {len(split_dataset['train'])}, Eval set: {len(split_dataset['test'])}")



### Step 7: SFT Training with Drive Checkpointing
Training arguments configured with Cosine Annealing, FP16, and checkpoint persistence to Google Drive every 250 steps.



In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

output_run_dir = f"{CHECKPOINT_DIR}/run_customer_support"

training_args = TrainingArguments(
    output_dir=output_run_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=1,
    logging_steps=25,
    save_steps=250,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=250,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("[*] Starting QLoRA fine-tuning...")
trainer.train()
print("[OK] Training complete.")



### Step 8: Save Adapters and Export GGUF / Merged Weights
Export LoRA weights to Google Drive and compile GGUF for local Ollama / Antigravity CLI deployment.



In [ ]:
final_adapter_path = f"{CHECKPOINT_DIR}/final_customer_support_adapter"
trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)

print(f"[OK] LoRA adapter saved to: {final_adapter_path}")

# Optional: Push to Hugging Face Hub
# trainer.model.push_to_hub("peiiaratef126-hub/customer-support-lora", token=HF_TOKEN)



In [ ]:
# Step 9: Test Inference with Fine-Tuned Model
from transformers import pipeline

prompt = (
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    f"<|im_start|>user\n[AppleSupport] My iPhone battery drops from 50% to 10% in 15 minutes. What should I do?<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("=== Generated Support Response ===")
print(response)

